In [ ]:
%load_ext autoreload
%autoreload 2


# Trading Competition Visualization

Load the latest Nash-DQN and LLQ SRE-DQN checkpoints and compare policies/rewards on the shared trading competition environment.

In [ ]:
from pathlib import Path
import os
import pickle
import sys
from datetime import datetime

import matplotlib.pyplot as plt
import numpy as np
import torch


def find_repo_root(start):
    start = Path(start).resolve()
    for path in [start, *start.parents]:
        if (path / 'AGENTS.md').exists() and (path / 'continuous_action_space').exists():
            return path
    raise RuntimeError('Could not locate repo root')

REPO_ROOT = find_repo_root(Path.cwd())
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

EXPERIMENT_DIR = REPO_ROOT / 'continuous_action_space' / 'trading_competition'
MODEL_ROOT = EXPERIMENT_DIR / 'pt_files'
EVAL_ROOT = EXPERIMENT_DIR / 'evaluation'
EVAL_ROOT.mkdir(parents=True, exist_ok=True)

np.set_printoptions(precision=4)
print('Repo root:', REPO_ROOT)
print('Model root:', MODEL_ROOT)
print('Using device:', 'cuda' if torch.cuda.is_available() else 'cpu')


## Imports And Checkpoint Helpers


In [ ]:
from continuous_action_space.trading_competition.experiment_config import (
    NUM_PLAYERS as num_players,
    sim_dict,
    norm_mean,
    norm_std,
    T,
    make_sim_obj,
    seed_everything,
    eps_slug,
    BASE_SEED,
    MAX_STEPS,
    LLQ_SRE_EPS_LIST,
    SRE_EPS_REG,
    SRE_DELTA_MIN,
    SRE_GAMMA,
    NET_KWARGS,
)
from continuous_action_space.trading_competition.visualization import (
    build_default_scenario_specs,
    collect_mixed_rewards,
    draw_heatmap,
    find_latest_model_dir,
    load_best_checkpoint_into_agent,
    loss_history_from_checkpoint,
    to_State_mesh_sre,
)
from continuous_action_space.locally_linear_quadratic.NashAgent_lib import NashNN
from continuous_action_space.locally_linear_quadratic.sre_agent import SreNN

sim_obj = make_sim_obj()


def build_nash_agent_from_config():
    return NashNN(**NET_KWARGS)


def build_llq_sre_agent_from_config():
    return SreNN(
        **NET_KWARGS,
        eps_reg=SRE_EPS_REG,
        delta_min=SRE_DELTA_MIN,
        gamma=SRE_GAMMA,
    )

print('Agents:', num_players, '| T=', sim_dict['T'].item(), '| dt=', sim_dict['dt'].item(), '| impact=', sim_obj.impact)

## Locate Latest Checkpoints


In [ ]:
EVAL_SEED = BASE_SEED + 1
RUN_TAG = datetime.now().strftime('%Y%m%d_%H%M')
EVAL_DIR = EVAL_ROOT / RUN_TAG
EVAL_DIR.mkdir(parents=True, exist_ok=True)

NASH_MODEL_DIR = find_latest_model_dir('nash', root_dir=str(MODEL_ROOT))
LLQ_SRE_MODEL_DIRS = {
    eps: find_latest_model_dir(f'llq_sre_eps_{eps_slug(eps)}', root_dir=str(MODEL_ROOT))
    for eps in LLQ_SRE_EPS_LIST
}

seed_everything(BASE_SEED)
print('Loaded Nash model dir:', NASH_MODEL_DIR)
print('Loaded LLQ SRE model dirs:')
for eps in LLQ_SRE_EPS_LIST:
    print(f'  eps={eps:g}:', LLQ_SRE_MODEL_DIRS[eps])
print('Evaluation dir:', EVAL_DIR)

## Load Agents


In [ ]:
nash_agent = build_nash_agent_from_config()
nash_best_checkpoint_path, nash_best_checkpoint = load_best_checkpoint_into_agent(nash_agent, NASH_MODEL_DIR)
nash_loss = loss_history_from_checkpoint(nash_best_checkpoint)

llq_sre_agents = {}
llq_sre_losses = {}
llq_sre_best_checkpoint_paths = {}
for eps in LLQ_SRE_EPS_LIST:
    llq_sre_agents[eps] = build_llq_sre_agent_from_config()
    checkpoint_path, checkpoint = load_best_checkpoint_into_agent(llq_sre_agents[eps], LLQ_SRE_MODEL_DIRS[eps])
    llq_sre_best_checkpoint_paths[eps] = checkpoint_path
    llq_sre_losses[eps] = loss_history_from_checkpoint(checkpoint)

print('Loaded all requested checkpoints.')

## Training Loss Curves


In [ ]:
def smooth(arr, window=100):
    arr = np.asarray(arr)
    if len(arr) < window:
        return arr
    kernel = np.ones(window) / window
    return np.convolve(arr, kernel, mode='valid')

plt.figure(figsize=(12, 6))
if nash_loss is not None:
    plt.plot(smooth(nash_loss), label='Nash-DQN', linewidth=2)
for eps, loss in llq_sre_losses.items():
    if loss is not None:
        plt.plot(smooth(loss), label=f'LLQ SRE eps={eps:g}', alpha=0.7)
plt.xlabel('Training iteration')
plt.ylabel('Smoothed total loss')
plt.legend(ncol=2)
plt.tight_layout()
plt.savefig(EVAL_DIR / 'training_loss_nash_llq_sre_smoothed.png')
plt.show()

## Policy Heatmaps


In [ ]:
heatmap_kwargs = dict(
    t_step=11,
    q_step=51,
    p_step=5,
    t_range=[0, 4.5],
    q_range=[-10, 10],
    p_range=[9.5, 10.5],
    n_agents=num_players,
    other_agent_inv=0,
    a_range=[-10, 10],
    T=T,
    norm_mean=norm_mean,
    norm_std=norm_std,
    is_numpy=False,
    norm_input=True,
)

for i_val in [0.2, 0.0, -0.2]:
    draw_heatmap(
        net=nash_agent,
        i_val=i_val,
        file_path=str(EVAL_DIR / f'policy_heatmap_nash_i{i_val:+.1f}.png'),
        figure_title=f'Nash-DQN Policy Heatmap | i={i_val:+.1f}',
        **heatmap_kwargs,
    )

for eps in [0.5, 1.0]:
    for i_val in [0.2, 0.0, -0.2]:
        draw_heatmap(
            net=llq_sre_agents[eps],
            i_val=i_val,
            file_path=str(EVAL_DIR / f'policy_heatmap_llq_sre_eps_{eps_slug(eps)}_i{i_val:+.1f}.png'),
            mesh_fn=to_State_mesh_sre,
            mesh_kwargs={'eps': eps},
            figure_title=f'LLQ SRE-DQN Corrected Action | eps={eps:g} | i={i_val:+.1f}',
            **heatmap_kwargs,
        )

## Evaluation Rollouts


In [ ]:
NUM_TRIALS = 50
IT_LIM = 500
EVAL_BATCH_SIZE = 2000

scenario_specs = build_default_scenario_specs(
    num_players=num_players,
    nash_agent=nash_agent,
    llq_sre_agents=llq_sre_agents,
    llq_eps_list=LLQ_SRE_EPS_LIST,
)

scenario_rewards = {}
for scenario_name, policy_specs in scenario_specs:
    scenario_rewards[scenario_name] = collect_mixed_rewards(
        sim_obj,
        norm_mean,
        norm_std,
        policy_specs,
        NUM_TRIALS,
        IT_LIM,
        seed=EVAL_SEED,
        desc=f'Collecting {scenario_name} rollouts...',
        eval_batch_size=EVAL_BATCH_SIZE,
    )
print('Done.')

## Reward Distributions And Save Results


In [ ]:
def agent0_mean_total_reward(rewards):
    return rewards.sum(dim=2)[..., 0].mean(dim=1).cpu().numpy()

scenario_agent0_rewards = {
    name: agent0_mean_total_reward(rewards)
    for name, rewards in scenario_rewards.items()
}

plt.figure(figsize=(12, 7))
for name, arr in scenario_agent0_rewards.items():
    plt.hist(arr, bins=20, alpha=0.35, label=name)
plt.xlabel('Agent 0 mean total reward')
plt.ylabel('Trial count')
plt.legend(fontsize=8)
plt.tight_layout()
plt.savefig(EVAL_DIR / 'reward_histogram_nash_llq_sre.png')
plt.show()

comparison_dict = {
    'nash_loss': nash_loss,
    'llq_sre_losses': llq_sre_losses,
    'scenario_rewards': scenario_rewards,
    'scenario_agent0_rewards': scenario_agent0_rewards,
    'config': {
        'BASE_SEED': BASE_SEED,
        'EVAL_SEED': EVAL_SEED,
        'MAX_STEPS': MAX_STEPS,
        'NUM_TRIALS': NUM_TRIALS,
        'IT_LIM': IT_LIM,
        'EVAL_BATCH_SIZE': EVAL_BATCH_SIZE,
        'LLQ_SRE_EPS_LIST': LLQ_SRE_EPS_LIST,
        'SRE_EPS_REG': SRE_EPS_REG,
        'nash_best_checkpoint_path': nash_best_checkpoint_path,
        'llq_sre_best_checkpoint_paths': llq_sre_best_checkpoint_paths,
        'evaluation_dir': str(EVAL_DIR),
    },
}

out_path = EVAL_DIR / 'nash_llq_sre_comparison.pickle'
with open(out_path, 'wb') as f:
    pickle.dump(comparison_dict, f)
print('Saved comparison results to:', out_path)